In [ ]:
%py
# Databricks PySpark script for d_product_revenue_clone table with masked invoice_number
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# Initialize Spark session
spark = SparkSession.builder.appName("Invoice Masking").getOrCreate()

# Load d_product_revenue table
try:
    d_product_revenue_df = spark.read.table("purgo_databricks.qa_1.d_product_revenue")
except Exception as e:
    print(f"Error loading d_product_revenue table: {e}")
    raise

# Mask the last 4 digits of the invoice_number in the dataframe
masked_df = d_product_revenue_df.withColumn(
    "invoice_number",
    when(
        col("invoice_number").isNotNull() & (col("invoice_number").rlike("^[0-9A-Za-z]+$")) & (col("invoice_number").rlength >= 4),
        col("invoice_number").substr(0, col("invoice_number").rlength - 4) + "****"
    ).otherwise(col("invoice_number"))
)

# Drop the clone table if it exists
spark.sql("DROP TABLE IF EXISTS purgo_databricks.qa_1.d_product_revenue_clone")

# Create a clone table with masked data
masked_df.write.format("delta").saveAsTable("purgo_databricks.qa_1.d_product_revenue_clone")

# Terminate Spark session
spark.stop()

